In [4]:
import warnings
warnings.simplefilter("ignore")

import pandas as pd
from lightkurve import search_lightcurve

# ============================================================
# LISTAR TODAS AS CURVAS DE LUZ DISPONÍVEIS
# ============================================================

targets = ["TIC 301455423"]

for target in targets:

    print("\n" + "="*100)
    print(f"BUSCANDO CURVAS DE LUZ PARA: {target}")
    print("="*100)

    # Busca todas as curvas de luz TESS
    search_result = search_lightcurve(target, mission="TESS")

    if len(search_result) == 0:
        print("Nenhuma curva de luz encontrada.")
        continue

    # Mostra a tabela completa do Lightkurve
    print("\nTabela retornada pelo Lightkurve:\n")
    print(search_result)

    # Converte para DataFrame
    df = search_result.table.to_pandas()

    print("\nNúmero total de curvas encontradas:", len(df))

    print("\nResumo das curvas disponíveis:\n")

    resumo = []

    for i, row in df.iterrows():

        setor = row["sequence_number"] if "sequence_number" in df.columns else "-"
        autor = row["author"] if "author" in df.columns else "-"
        missao = row["mission"] if "mission" in df.columns else "-"
        exptime = row["exptime"] if "exptime" in df.columns else "-"
        produto = row["productFilename"] if "productFilename" in df.columns else "-"

        resumo.append({
            "LC": i,
            "Setor": setor,
            "Autor": autor,
            "Missão": missao,
            "Exposição (s)": exptime,
            "Arquivo": produto
        })

    resumo = pd.DataFrame(resumo)

    print(resumo.to_string(index=False))

    print("\n" + "="*100)
    print("SETORES DISPONÍVEIS")
    print("="*100)

    print(sorted(resumo["Setor"].unique()))

    print("\n" + "="*100)
    print("AUTORES DISPONÍVEIS")
    print("="*100)

    print(sorted(resumo["Autor"].unique()))

    print("\n" + "="*100)
    print("TEMPOS DE EXPOSIÇÃO DISPONÍVEIS")
    print("="*100)

    print(sorted(resumo["Exposição (s)"].unique()))

    print("\n" + "="*100)
    print("CURVAS POR AUTOR")
    print("="*100)

    for autor in sorted(resumo["Autor"].unique()):

        print(f"\nAutor: {autor}")

        df_autor = resumo[resumo["Autor"] == autor]

        print(df_autor[["LC",
                        "Setor",
                        "Exposição (s)",
                        "Arquivo"]].to_string(index=False))

print("\nBusca finalizada.")


BUSCANDO CURVAS DE LUZ PARA: TIC 301455423

Tabela retornada pelo Lightkurve:

SearchResult containing 18 data products.

 #      mission     year       author      exptime target_name distance
                                              s                 arcsec 
--- --------------- ---- ----------------- ------- ----------- --------
  0  TESS Sector 66 2023              SPOC     120   301455423      0.0
  1  TESS Sector 93 2025              SPOC     120   301455423      0.0
  2 TESS Sector 100 2026              SPOC     120   301455423      0.0
  3 TESS Sector 101 2026              SPOC     120   301455423      0.0
  4 TESS Sector 102 2026              SPOC     120   301455423      0.0
  5 TESS Sector 103 2026              SPOC     120   301455423      0.0
  6  TESS Sector 66 2023         TESS-SPOC     200   301455423      0.0
  7  TESS Sector 12 2019               QLP    1800   301455423      0.0
  8  TESS Sector 13 2019               QLP    1800   301455423      0.0
  9  TESS Sec

In [5]:
import warnings
warnings.simplefilter("ignore")

import numpy as np
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve

%matplotlib qt

# ==========================================================
# ALVO
# ==========================================================

target = "TIC 301455423"

# ==========================================================
# EFEMÉRIDE
# ==========================================================

T0_BJD = 2459387.102253
P = 1.0664194                     # dias
duration_hours = 2.106    # horas

# Lightkurve trabalha em BTJD
T0 = T0_BJD - 2457000

duration = duration_hours / 24.0

# ==========================================================
# BAIXAR SETOR 100 SPOC
# ==========================================================

search = search_lightcurve(
    target,
    mission="TESS",
    author="SPOC"
)

indice = None

for i, row in enumerate(search.table):

    if ("Sector 100" in row["mission"]) and (row["exptime"] == 120):

        indice = i
        break

if indice is None:
    raise Exception("Setor 100 não encontrado.")

print(search[indice])

lc = search[indice].download()

# ==========================================================
# LIMPEZA
# ==========================================================

lc = lc.remove_nans()
lc = lc.remove_outliers()
lc = lc[lc.quality == 0]
lc = lc.normalize().remove_nans()

time = lc.time.value
flux = lc.flux.value

# ==========================================================
# CALCULA TODOS OS TRÂNSITOS
# ==========================================================

tmin = np.min(time)
tmax = np.max(time)

n_ini = int(np.floor((tmin-T0)/P))
n_fim = int(np.ceil((tmax-T0)/P))

transitos = []

for n in range(n_ini,n_fim+1):

    tc = T0 + n*P

    if tmin <= tc <= tmax:
        transitos.append(tc)

print("\nNúmero de trânsitos encontrados:",len(transitos))

# ==========================================================
# CURVA COMPLETA
# ==========================================================

plt.figure(figsize=(16,6))

plt.plot(
    time,
    flux,
    ".",
    color="black",
    ms=1
)

for i,tc in enumerate(transitos):

    plt.axvline(tc,
                color="red",
                lw=1.5)

    plt.axvspan(
        tc-duration/2,
        tc+duration/2,
        color="red",
        alpha=0.20
    )

    plt.text(
        tc,
        np.max(flux),
        f"T{i+1}",
        rotation=90,
        color="red",
        fontsize=10,
        ha="center"
    )

plt.xlabel("BTJD")
plt.ylabel("Normalized Flux")
plt.title("TOI 3326.01 - TIC 301455423\nTESS SPOC - Setor 100")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ==========================================================
# ZOOM DE CADA TRÂNSITO
# ==========================================================

for i,tc in enumerate(transitos):

    janela = 0.8

    mask = (
        (time>=tc-janela) &
        (time<=tc+janela)
    )

    plt.figure(figsize=(9,4))

    plt.plot(
        time[mask],
        flux[mask],
        ".k",
        ms=3
    )

    plt.axvline(
        tc,
        color="red",
        lw=2
    )

    plt.axvspan(
        tc-duration/2,
        tc+duration/2,
        color="red",
        alpha=0.25
    )

    plt.title(f"Trânsito {i+1}")

    plt.xlabel("BTJD")
    plt.ylabel("Flux")

    plt.grid(alpha=0.3)

    plt.tight_layout()

    plt.show()

# ==========================================================
# TABELA
# ==========================================================

print("\n==============================")
print("TRÂNSITOS NO SETOR 100")
print("==============================")

for i,tc in enumerate(transitos):

    print(f"{i+1:2d}   Centro = {tc:.6f} BTJD")

SearchResult containing 1 data products.

 #      mission     year author exptime target_name distance
                                   s                 arcsec 
--- --------------- ---- ------ ------- ----------- --------
  0 TESS Sector 100 2026   SPOC     120   301455423      0.0

Número de trânsitos encontrados: 23

TRÂNSITOS NO SETOR 100
 1   Centro = 4075.244163 BTJD
 2   Centro = 4076.310583 BTJD
 3   Centro = 4077.377002 BTJD
 4   Centro = 4078.443421 BTJD
 5   Centro = 4079.509841 BTJD
 6   Centro = 4080.576260 BTJD
 7   Centro = 4081.642680 BTJD
 8   Centro = 4082.709099 BTJD
 9   Centro = 4083.775518 BTJD
10   Centro = 4084.841938 BTJD
11   Centro = 4085.908357 BTJD
12   Centro = 4086.974777 BTJD
13   Centro = 4088.041196 BTJD
14   Centro = 4089.107615 BTJD
15   Centro = 4090.174035 BTJD
16   Centro = 4091.240454 BTJD
17   Centro = 4092.306874 BTJD
18   Centro = 4093.373293 BTJD
19   Centro = 4094.439712 BTJD
20   Centro = 4095.506132 BTJD
21   Centro = 4096.572551 BTJD
22

In [7]:
import warnings
warnings.simplefilter("ignore")

import numpy as np
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve

%matplotlib qt

# ==========================================================
# ALVO
# ==========================================================

target = "TIC 301455423"

# ==========================================================
# EFEMÉRIDE
# ==========================================================

T0_BJD = 2459387.102253
P = 1.0664194                 # dias
duration_hours = 2.106        # horas

# Lightkurve trabalha em BTJD
T0 = T0_BJD - 2457000
duration = duration_hours / 24.0

# ==========================================================
# BAIXAR SETOR 100 SPOC
# ==========================================================

search = search_lightcurve(
    target,
    mission="TESS",
    author="SPOC"
)

indice = None

for i, row in enumerate(search.table):
    if ("Sector 100" in row["mission"]) and (row["exptime"] == 120):
        indice = i
        break

if indice is None:
    raise Exception("Setor 100 não encontrado.")

print(search[indice])

lc = search[indice].download()

# ==========================================================
# LIMPEZA
# ==========================================================

lc = lc.remove_nans()
lc = lc.remove_outliers()
lc = lc[lc.quality == 0]
lc = lc.normalize().remove_nans()

time = lc.time.value
flux = lc.flux.value

# ==========================================================
# CALCULA TODOS OS TRÂNSITOS
# ==========================================================

tmin = np.min(time)
tmax = np.max(time)

n_ini = int(np.floor((tmin - T0) / P))
n_fim = int(np.ceil((tmax - T0) / P))

transitos = []

for n in range(n_ini, n_fim + 1):
    tc = T0 + n * P
    if tmin <= tc <= tmax:
        transitos.append(tc)

print("\nNúmero de trânsitos encontrados:", len(transitos))

# ==========================================================
# GRÁFICO 1 - CURVA COMPLETA + TRÂNSITOS
# ==========================================================

plt.figure(figsize=(16, 6))

plt.plot(time, flux, ".-", color="black", ms=2, lw=0.5)

for i, tc in enumerate(transitos):

    plt.axvline(tc, color="red", lw=1.5)

    plt.axvspan(
        tc - duration / 2,
        tc + duration / 2,
        color="red",
        alpha=0.20
    )

    plt.text(
        tc,
        np.max(flux),
        f"T{i+1}",
        rotation=90,
        color="red",
        fontsize=10,
        ha="center"
    )

plt.xlabel("BTJD")
plt.ylabel("Normalized Flux")
plt.title("TOI 3326.01 - TIC 301455423\nTESS SPOC - Setor 100")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ==========================================================
# GRÁFICO 2 - TODOS OS TRÂNSITOS SOBREPOSTOS
# ==========================================================

janela = 0.8   # dias em torno do centro

plt.figure(figsize=(10, 6))

for i, tc in enumerate(transitos):

    mask = (time >= tc - janela) & (time <= tc + janela)

    plt.plot(
        (time[mask] - tc) * 24.0,   # horas relativas ao centro
        flux[mask],
        ".-",
        ms=3,
        lw=0.5,
        label=f"T{i+1}"
    )

plt.axvline(0, color="red", lw=1.5)

plt.axvspan(
    -duration_hours / 2,
    +duration_hours / 2,
    color="red",
    alpha=0.15
)

plt.xlabel("Horas desde o centro do trânsito")
plt.ylabel("Normalized Flux")
plt.title("Trânsitos observados sobrepostos - Setor 100")
plt.grid(alpha=0.3)
plt.legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.show()

# ==========================================================
# TABELA
# ==========================================================

print("\n==============================")
print("TRÂNSITOS NO SETOR 100")
print("==============================")

for i, tc in enumerate(transitos):
    print(f"{i+1:2d}   Centro = {tc:.6f} BTJD")

SearchResult containing 1 data products.

 #      mission     year author exptime target_name distance
                                   s                 arcsec 
--- --------------- ---- ------ ------- ----------- --------
  0 TESS Sector 100 2026   SPOC     120   301455423      0.0

Número de trânsitos encontrados: 23

TRÂNSITOS NO SETOR 100
 1   Centro = 4075.244163 BTJD
 2   Centro = 4076.310583 BTJD
 3   Centro = 4077.377002 BTJD
 4   Centro = 4078.443421 BTJD
 5   Centro = 4079.509841 BTJD
 6   Centro = 4080.576260 BTJD
 7   Centro = 4081.642680 BTJD
 8   Centro = 4082.709099 BTJD
 9   Centro = 4083.775518 BTJD
10   Centro = 4084.841938 BTJD
11   Centro = 4085.908357 BTJD
12   Centro = 4086.974777 BTJD
13   Centro = 4088.041196 BTJD
14   Centro = 4089.107615 BTJD
15   Centro = 4090.174035 BTJD
16   Centro = 4091.240454 BTJD
17   Centro = 4092.306874 BTJD
18   Centro = 4093.373293 BTJD
19   Centro = 4094.439712 BTJD
20   Centro = 4095.506132 BTJD
21   Centro = 4096.572551 BTJD
22

In [8]:
import warnings
warnings.simplefilter("ignore")

import numpy as np
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve, LightCurveCollection

%matplotlib qt

# ==========================================================
# ALVO
# ==========================================================

target = "TIC 301455423"

# ==========================================================
# EFEMÉRIDE
# ==========================================================

T0_BJD = 2459387.102253
P = 1.0664194                 # dias
duration_hours = 2.106        # horas

# Lightkurve trabalha em BTJD
T0 = T0_BJD - 2457000
duration = duration_hours / 24.0

# ==========================================================
# SETORES A BAIXAR
# ==========================================================

setores = [101, 102, 103]

# ==========================================================
# BAIXAR SETORES SPOC
# ==========================================================

search = search_lightcurve(
    target,
    mission="TESS",
    author="SPOC"
)

lc_list = []

for s in setores:

    indice = None

    for i, row in enumerate(search.table):
        if (f"Sector {s}" in row["mission"]) and (row["exptime"] == 120):
            indice = i
            break

    if indice is None:
        print(f"Setor {s} não encontrado.")
        continue

    print(search[indice])

    lc_s = search[indice].download()

    # limpeza por setor
    lc_s = lc_s.remove_nans()
    lc_s = lc_s.remove_outliers()
    lc_s = lc_s[lc_s.quality == 0]
    lc_s = lc_s.normalize().remove_nans()

    lc_list.append(lc_s)

if len(lc_list) == 0:
    raise Exception("Nenhum setor encontrado.")

# junta todos os setores numa curva só
lc = LightCurveCollection(lc_list).stitch()

time = lc.time.value
flux = lc.flux.value

# ==========================================================
# CALCULA TODOS OS TRÂNSITOS
# ==========================================================

tmin = np.min(time)
tmax = np.max(time)

n_ini = int(np.floor((tmin - T0) / P))
n_fim = int(np.ceil((tmax - T0) / P))

transitos = []

for n in range(n_ini, n_fim + 1):
    tc = T0 + n * P
    if tmin <= tc <= tmax:
        # garante que há dados perto do centro (evita marcar em gaps)
        if np.any(np.abs(time - tc) < duration / 2):
            transitos.append(tc)

print("\nNúmero de trânsitos encontrados:", len(transitos))

# ==========================================================
# GRÁFICO 1 - CURVA COMPLETA + TRÂNSITOS
# ==========================================================

plt.figure(figsize=(16, 6))

plt.plot(time, flux, ".-", color="black", ms=2, lw=0.5)

for i, tc in enumerate(transitos):

    plt.axvline(tc, color="red", lw=1.5)

    plt.axvspan(
        tc - duration / 2,
        tc + duration / 2,
        color="red",
        alpha=0.20
    )

    plt.text(
        tc,
        np.max(flux),
        f"T{i+1}",
        rotation=90,
        color="red",
        fontsize=10,
        ha="center"
    )

plt.xlabel("BTJD")
plt.ylabel("Normalized Flux")
plt.title(f"TOI 3326.01 - TIC 301455423\nTESS SPOC - Setores {setores}")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ==========================================================
# GRÁFICO 2 - TODOS OS TRÂNSITOS SOBREPOSTOS
# ==========================================================

janela = 0.8   # dias em torno do centro

plt.figure(figsize=(10, 6))

for i, tc in enumerate(transitos):

    mask = (time >= tc - janela) & (time <= tc + janela)

    plt.plot(
        (time[mask] - tc) * 24.0,   # horas relativas ao centro
        flux[mask],
        ".-",
        ms=3,
        lw=0.5,
        label=f"T{i+1}"
    )

plt.axvline(0, color="red", lw=1.5)

plt.axvspan(
    -duration_hours / 2,
    +duration_hours / 2,
    color="red",
    alpha=0.15
)

plt.xlabel("Horas desde o centro do trânsito")
plt.ylabel("Normalized Flux")
plt.title(f"Trânsitos observados sobrepostos - Setores {setores}")
plt.grid(alpha=0.3)
plt.legend(fontsize=7, ncol=3)

plt.tight_layout()
plt.show()

# ==========================================================
# TABELA
# ==========================================================

print("\n==============================")
print("TRÂNSITOS - SETORES", setores)
print("==============================")

for i, tc in enumerate(transitos):
    print(f"{i+1:2d}   Centro = {tc:.6f} BTJD")

SearchResult containing 1 data products.

 #      mission     year author exptime target_name distance
                                   s                 arcsec 
--- --------------- ---- ------ ------- ----------- --------
  0 TESS Sector 101 2026   SPOC     120   301455423      0.0
SearchResult containing 1 data products.

 #      mission     year author exptime target_name distance
                                   s                 arcsec 
--- --------------- ---- ------ ------- ----------- --------
  0 TESS Sector 102 2026   SPOC     120   301455423      0.0
SearchResult containing 1 data products.

 #      mission     year author exptime target_name distance
                                   s                 arcsec 
--- --------------- ---- ------ ------- ----------- --------
  0 TESS Sector 103 2026   SPOC     120   301455423      0.0

Número de trânsitos encontrados: 60

TRÂNSITOS - SETORES [101, 102, 103]
 1   Centro = 4101.904648 BTJD
 2   Centro = 4102.971068 BTJD
 3   C

In [9]:
import warnings
warnings.simplefilter("ignore")

import numpy as np
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve

%matplotlib qt

# ==========================================================
# ALVO
# ==========================================================

target = "TIC 301455423"

# ==========================================================
# EFEMÉRIDE
# ==========================================================

T0_BJD = 2459387.102253
P = 1.0664194                 # dias
duration_hours = 2.106        # horas

# Lightkurve trabalha em BTJD
T0 = T0_BJD - 2457000
duration = duration_hours / 24.0

# ==========================================================
# SETORES A BAIXAR
# ==========================================================

setores = [101, 102, 103]

# ==========================================================
# BUSCA SPOC
# ==========================================================

search = search_lightcurve(
    target,
    mission="TESS",
    author="SPOC"
)

# ==========================================================
# LOOP POR SETOR - CADA UM COM SEUS GRÁFICOS
# ==========================================================

for s in setores:

    # --------- localiza e baixa o setor ---------
    indice = None
    for i, row in enumerate(search.table):
        if (f"Sector {s}" in row["mission"]) and (row["exptime"] == 120):
            indice = i
            break

    if indice is None:
        print(f"Setor {s} não encontrado.")
        continue

    print(search[indice])

    lc = search[indice].download()

    # --------- limpeza ---------
    lc = lc.remove_nans()
    lc = lc.remove_outliers()
    lc = lc[lc.quality == 0]
    lc = lc.normalize().remove_nans()

    time = lc.time.value
    flux = lc.flux.value

    # --------- trânsitos deste setor ---------
    tmin = np.min(time)
    tmax = np.max(time)

    n_ini = int(np.floor((tmin - T0) / P))
    n_fim = int(np.ceil((tmax - T0) / P))

    transitos = []
    for n in range(n_ini, n_fim + 1):
        tc = T0 + n * P
        if tmin <= tc <= tmax:
            if np.any(np.abs(time - tc) < duration / 2):
                transitos.append(tc)

    print(f"\nSetor {s} - trânsitos encontrados:", len(transitos))

    # ======================================================
    # GRÁFICO 1 - CURVA COMPLETA + TRÂNSITOS
    # ======================================================

    plt.figure(figsize=(16, 6))

    plt.plot(time, flux, ".-", color="black", ms=2, lw=0.5)

    for i, tc in enumerate(transitos):

        plt.axvline(tc, color="red", lw=1.5)

        plt.axvspan(
            tc - duration / 2,
            tc + duration / 2,
            color="red",
            alpha=0.20
        )

        plt.text(
            tc,
            np.max(flux),
            f"T{i+1}",
            rotation=90,
            color="red",
            fontsize=10,
            ha="center"
        )

    plt.xlabel("BTJD")
    plt.ylabel("Normalized Flux")
    plt.title(f"TOI 3326.01 - TIC 301455423\nTESS SPOC - Setor {s}")
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # ======================================================
    # GRÁFICO 2 - TRÂNSITOS SOBREPOSTOS DESTE SETOR
    # ======================================================

    janela = 0.8   # dias em torno do centro

    plt.figure(figsize=(10, 6))

    for i, tc in enumerate(transitos):

        mask = (time >= tc - janela) & (time <= tc + janela)

        plt.plot(
            (time[mask] - tc) * 24.0,   # horas relativas ao centro
            flux[mask],
            ".-",
            ms=3,
            lw=0.5,
            label=f"T{i+1}"
        )

    plt.axvline(0, color="red", lw=1.5)

    plt.axvspan(
        -duration_hours / 2,
        +duration_hours / 2,
        color="red",
        alpha=0.15
    )

    plt.xlabel("Horas desde o centro do trânsito")
    plt.ylabel("Normalized Flux")
    plt.title(f"Trânsitos sobrepostos - Setor {s}")
    plt.grid(alpha=0.3)
    plt.legend(fontsize=8, ncol=2)

    plt.tight_layout()
    plt.show()

    # ======================================================
    # TABELA DESTE SETOR
    # ======================================================

    print(f"\n==============================")
    print(f"TRÂNSITOS - SETOR {s}")
    print(f"==============================")

    for i, tc in enumerate(transitos):
        print(f"{i+1:2d}   Centro = {tc:.6f} BTJD")

SearchResult containing 1 data products.

 #      mission     year author exptime target_name distance
                                   s                 arcsec 
--- --------------- ---- ------ ------- ----------- --------
  0 TESS Sector 101 2026   SPOC     120   301455423      0.0

Setor 101 - trânsitos encontrados: 20

TRÂNSITOS - SETOR 101
 1   Centro = 4101.904648 BTJD
 2   Centro = 4102.971068 BTJD
 3   Centro = 4104.037487 BTJD
 4   Centro = 4105.103906 BTJD
 5   Centro = 4106.170326 BTJD
 6   Centro = 4107.236745 BTJD
 7   Centro = 4108.303165 BTJD
 8   Centro = 4109.369584 BTJD
 9   Centro = 4110.436003 BTJD
10   Centro = 4111.502423 BTJD
11   Centro = 4114.701681 BTJD
12   Centro = 4115.768100 BTJD
13   Centro = 4116.834520 BTJD
14   Centro = 4117.900939 BTJD
15   Centro = 4118.967359 BTJD
16   Centro = 4120.033778 BTJD
17   Centro = 4121.100197 BTJD
18   Centro = 4122.166617 BTJD
19   Centro = 4123.233036 BTJD
20   Centro = 4124.299456 BTJD
SearchResult containing 1 data p

In [11]:
import warnings
warnings.simplefilter("ignore")

import numpy as np
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve, LightCurveCollection

%matplotlib qt

# ==========================================================
# ALVO
# ==========================================================

target = "TIC 301455423"

# ==========================================================
# EFEMÉRIDE
# ==========================================================

T0_BJD = 2459387.102253
P = 1.0664194                 # dias
duration_hours = 2.106        # horas

# Lightkurve trabalha em BTJD
T0 = T0_BJD - 2457000
duration = duration_hours / 24.0

# ==========================================================
# BUSCA SPOC - TODAS AS CURVAS 120s
# ==========================================================

search = search_lightcurve(
    target,
    mission="TESS",
    author="SPOC"
)

lc_list = []

for i, row in enumerate(search.table):

    if row["exptime"] != 120:
        continue

    print(search[i])

    lc_s = search[i].download()

    # limpeza por setor
    lc_s = lc_s.remove_nans()
    lc_s = lc_s.remove_outliers()
    lc_s = lc_s[lc_s.quality == 0]
    lc_s = lc_s.normalize().remove_nans()

    lc_list.append(lc_s)

if len(lc_list) == 0:
    raise Exception("Nenhuma curva 120s encontrada.")

# junta todos os setores numa curva só
lc = LightCurveCollection(lc_list).stitch()

time = lc.time.value
flux = lc.flux.value

print("\nTotal de setores baixados:", len(lc_list))

# ==========================================================
# GRÁFICO 1 - TODAS AS CURVAS JUNTAS
# ==========================================================

plt.figure(figsize=(16, 6))

plt.plot(time, flux, ".-", color="black", ms=2, lw=0.5)

plt.xlabel("BTJD")
plt.ylabel("Normalized Flux")
plt.title("TOI 3326.01 - TIC 301455423\nTESS SPOC - Todas as curvas")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ==========================================================
# PHASE FOLD
# ==========================================================

lc_fold = lc.fold(period=P, epoch_time=T0)

phase = lc_fold.phase.value          # em dias
flux_fold = lc_fold.flux.value

# ==========================================================
# GRÁFICO 2 - CURVA DOBRADA EM FASE
# ==========================================================

plt.figure(figsize=(10, 6))

plt.plot(
    phase * 24.0,      # horas desde o centro do trânsito
    flux_fold,
    ".",
    color="black",
    ms=2
)

plt.axvline(0, color="red", lw=1.5)

plt.axvspan(
    -duration_hours / 2,
    +duration_hours / 2,
    color="red",
    alpha=0.15
)

plt.xlabel("Horas desde o centro do trânsito")
plt.ylabel("Normalized Flux")
plt.title("Phase Fold - Todas as curvas")
plt.grid(alpha=0.3)

# zoom em torno do trânsito
plt.xlim(-6, 6)

plt.tight_layout()
plt.show()

SearchResult containing 1 data products.

 #     mission     year author exptime target_name distance
                                  s                 arcsec 
--- -------------- ---- ------ ------- ----------- --------
  0 TESS Sector 66 2023   SPOC     120   301455423      0.0
SearchResult containing 1 data products.

 #     mission     year author exptime target_name distance
                                  s                 arcsec 
--- -------------- ---- ------ ------- ----------- --------
  0 TESS Sector 93 2025   SPOC     120   301455423      0.0
SearchResult containing 1 data products.

 #      mission     year author exptime target_name distance
                                   s                 arcsec 
--- --------------- ---- ------ ------- ----------- --------
  0 TESS Sector 100 2026   SPOC     120   301455423      0.0
SearchResult containing 1 data products.

 #      mission     year author exptime target_name distance
                                   s           

In [12]:
import warnings
warnings.simplefilter("ignore")

import numpy as np
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve, LightCurveCollection

%matplotlib qt

# ==========================================================
# ALVO
# ==========================================================

target = "TIC 301455423"

# ==========================================================
# EFEMÉRIDE
# ==========================================================

T0_BJD = 2459387.102253
P = 1.0664194                 # dias
duration_hours = 2.106        # horas

# Lightkurve trabalha em BTJD
T0 = T0_BJD - 2457000
duration = duration_hours / 24.0

# ==========================================================
# SETORES DESTE ANO
# ==========================================================

setores = [100, 101, 102, 103]

# ==========================================================
# BUSCA SPOC - SÓ OS SETORES ESCOLHIDOS (120s)
# ==========================================================

search = search_lightcurve(
    target,
    mission="TESS",
    author="SPOC"
)

lc_list = []

for s in setores:

    indice = None
    for i, row in enumerate(search.table):
        if (f"Sector {s}" in row["mission"]) and (row["exptime"] == 120):
            indice = i
            break

    if indice is None:
        print(f"Setor {s} não encontrado.")
        continue

    print(search[indice])

    lc_s = search[indice].download()

    # limpeza por setor
    lc_s = lc_s.remove_nans()
    lc_s = lc_s.remove_outliers()
    lc_s = lc_s[lc_s.quality == 0]
    lc_s = lc_s.normalize().remove_nans()

    lc_list.append(lc_s)

if len(lc_list) == 0:
    raise Exception("Nenhuma curva encontrada.")

# junta os setores numa curva só
lc = LightCurveCollection(lc_list).stitch()

time = lc.time.value
flux = lc.flux.value

print("\nTotal de setores baixados:", len(lc_list))

# ==========================================================
# CALCULA TODOS OS TRÂNSITOS
# ==========================================================

tmin = np.min(time)
tmax = np.max(time)

n_ini = int(np.floor((tmin - T0) / P))
n_fim = int(np.ceil((tmax - T0) / P))

transitos = []
for n in range(n_ini, n_fim + 1):
    tc = T0 + n * P
    if tmin <= tc <= tmax:
        # só marca se houver dados perto do centro (evita gaps)
        if np.any(np.abs(time - tc) < duration / 2):
            transitos.append(tc)

print("Número de trânsitos encontrados:", len(transitos))

# ==========================================================
# GRÁFICO 1 - TODAS AS CURVAS JUNTAS + TRÂNSITOS
# ==========================================================

plt.figure(figsize=(16, 6))

plt.plot(time, flux, ".-", color="black", ms=2, lw=0.5)

for i, tc in enumerate(transitos):

    plt.axvline(tc, color="red", lw=1.5)

    plt.axvspan(
        tc - duration / 2,
        tc + duration / 2,
        color="red",
        alpha=0.20
    )

    plt.text(
        tc,
        np.max(flux),
        f"T{i+1}",
        rotation=90,
        color="red",
        fontsize=9,
        ha="center"
    )

plt.xlabel("BTJD")
plt.ylabel("Normalized Flux")
plt.title(f"TOI 3326.01 - TIC 301455423\nTESS SPOC - Setores {setores}")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ==========================================================
# PHASE FOLD
# ==========================================================

lc_fold = lc.fold(period=P, epoch_time=T0)

phase = lc_fold.phase.value          # em dias
flux_fold = lc_fold.flux.value

# ==========================================================
# GRÁFICO 2 - CURVA DOBRADA EM FASE
# ==========================================================

plt.figure(figsize=(10, 6))

plt.plot(
    phase * 24.0,      # horas desde o centro do trânsito
    flux_fold,
    ".",
    color="black",
    ms=2
)

plt.axvline(0, color="red", lw=1.5)

plt.axvspan(
    -duration_hours / 2,
    +duration_hours / 2,
    color="red",
    alpha=0.15
)

plt.xlabel("Horas desde o centro do trânsito")
plt.ylabel("Normalized Flux")
plt.title(f"Phase Fold - Setores {setores}")
plt.grid(alpha=0.3)

plt.xlim(-6, 6)   # zoom no trânsito; remova para ver a fase inteira

plt.tight_layout()
plt.show()

# ==========================================================
# TABELA - ONDE ESTÁ CADA TRÂNSITO
# ==========================================================

print("\n==============================")
print("TRÂNSITOS - SETORES", setores)
print("==============================")

for i, tc in enumerate(transitos):
    tc_bjd = tc + 2457000
    print(f"T{i+1:2d}   BTJD = {tc:.6f}   |   BJD = {tc_bjd:.6f}")

SearchResult containing 1 data products.

 #      mission     year author exptime target_name distance
                                   s                 arcsec 
--- --------------- ---- ------ ------- ----------- --------
  0 TESS Sector 100 2026   SPOC     120   301455423      0.0
SearchResult containing 1 data products.

 #      mission     year author exptime target_name distance
                                   s                 arcsec 
--- --------------- ---- ------ ------- ----------- --------
  0 TESS Sector 101 2026   SPOC     120   301455423      0.0
SearchResult containing 1 data products.

 #      mission     year author exptime target_name distance
                                   s                 arcsec 
--- --------------- ---- ------ ------- ----------- --------
  0 TESS Sector 102 2026   SPOC     120   301455423      0.0
SearchResult containing 1 data products.

 #      mission     year author exptime target_name distance
                                   s   

In [13]:
import warnings
warnings.simplefilter("ignore")

import numpy as np
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve, LightCurveCollection

%matplotlib qt

# ==========================================================
# ALVO
# ==========================================================

target = "TIC 301455423"

# ==========================================================
# EFEMÉRIDE
# ==========================================================

T0_BJD = 2459387.102253
P = 1.0664194                 # dias
duration_hours = 2.106        # horas

T0 = T0_BJD - 2457000
duration = duration_hours / 24.0

# ==========================================================
# SETORES DESTE ANO
# ==========================================================

setores = [100, 101, 102, 103]

# ==========================================================
# BUSCA SPOC
# ==========================================================

search = search_lightcurve(
    target,
    mission="TESS",
    author="SPOC"
)

lc_list = []

for s in setores:

    indice = None
    for i, row in enumerate(search.table):
        if (f"Sector {s}" in row["mission"]) and (row["exptime"] == 120):
            indice = i
            break

    if indice is None:
        print(f"Setor {s} não encontrado.")
        continue

    lc_s = search[indice].download()
    lc_s = lc_s.remove_nans()
    lc_s = lc_s.remove_outliers()
    lc_s = lc_s[lc_s.quality == 0]
    lc_s = lc_s.normalize().remove_nans()

    lc_list.append(lc_s)

if len(lc_list) == 0:
    raise Exception("Nenhuma curva encontrada.")

lc = LightCurveCollection(lc_list).stitch()

time = lc.time.value
flux = lc.flux.value

# ==========================================================
# TRÂNSITOS INDIVIDUAIS
# ==========================================================

tmin = np.min(time)
tmax = np.max(time)

n_ini = int(np.floor((tmin - T0) / P))
n_fim = int(np.ceil((tmax - T0) / P))

transitos = []
for n in range(n_ini, n_fim + 1):
    tc = T0 + n * P
    if tmin <= tc <= tmax:
        if np.any(np.abs(time - tc) < duration / 2):
            transitos.append(tc)

print("Número de trânsitos encontrados:", len(transitos))

# ==========================================================
# GRÁFICO ÚNICO - TODOS OS TRÂNSITOS SOBREPOSTOS
# ==========================================================

janela = 0.25   # dias em torno do centro

plt.figure(figsize=(10, 6))

for i, tc in enumerate(transitos):

    mask = (time >= tc - janela) & (time <= tc + janela)

    plt.plot(
        (time[mask] - tc) * 24.0,   # horas relativas ao centro
        flux[mask],
        ".-",
        ms=3,
        lw=0.5,
        label=f"T{i+1}"
    )

plt.axvline(0, color="red", lw=1.5)

plt.axvspan(
    -duration_hours / 2,
    +duration_hours / 2,
    color="red",
    alpha=0.15
)

plt.xlabel("Horas desde o centro do trânsito")
plt.ylabel("Normalized Flux")
plt.title(f"Todos os trânsitos - Setores {setores}")
plt.grid(alpha=0.3)
plt.legend(fontsize=7, ncol=3)

plt.tight_layout()
plt.show()

Número de trânsitos encontrados: 81


In [14]:
import warnings
warnings.simplefilter("ignore")

import numpy as np
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve, LightCurveCollection

%matplotlib qt

# ==========================================================
# ESTILO
# ==========================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.facecolor": "#fafafa",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.8,
    "axes.axisbelow": True,
    "figure.facecolor": "white",
})

ACCENT = "#e63946"      # vermelho trânsito
POINT  = "#1d3557"      # azul escuro pontos

# ==========================================================
# ALVO / EFEMÉRIDE
# ==========================================================

target = "TIC 301455423"

T0_BJD = 2459387.102253
P = 1.0664194                 # dias
duration_hours = 2.106        # horas

T0 = T0_BJD - 2457000
duration = duration_hours / 24.0

setores = [100, 101, 102, 103]

# ==========================================================
# BUSCA / DOWNLOAD
# ==========================================================

search = search_lightcurve(target, mission="TESS", author="SPOC")

lc_dict = {}

for s in setores:

    indice = None
    for i, row in enumerate(search.table):
        if (f"Sector {s}" in row["mission"]) and (row["exptime"] == 120):
            indice = i
            break

    if indice is None:
        print(f"Setor {s} não encontrado.")
        continue

    lc_s = search[indice].download()
    lc_s = lc_s.remove_nans()
    lc_s = lc_s.remove_outliers()
    lc_s = lc_s[lc_s.quality == 0]
    lc_s = lc_s.normalize().remove_nans()

    lc_dict[s] = lc_s

if len(lc_dict) == 0:
    raise Exception("Nenhuma curva encontrada.")


# helper: trânsitos dentro de um vetor de tempo
def transitos_em(time):
    n_ini = int(np.floor((np.min(time) - T0) / P))
    n_fim = int(np.ceil((np.max(time) - T0) / P))
    tt = []
    for n in range(n_ini, n_fim + 1):
        tc = T0 + n * P
        if np.min(time) <= tc <= np.max(time):
            if np.any(np.abs(time - tc) < duration / 2):
                tt.append(tc)
    return tt

# ==========================================================
# FIGURA 1 - PAINEL POR SETOR
# ==========================================================

n = len(lc_dict)
fig, axes = plt.subplots(n, 1, figsize=(15, 3.2 * n), sharey=True)

if n == 1:
    axes = [axes]

for ax, (s, lc_s) in zip(axes, lc_dict.items()):

    time = lc_s.time.value
    flux = lc_s.flux.value

    ax.plot(time, flux, ".", color=POINT, ms=1.5, alpha=0.7)

    for i, tc in enumerate(transitos_em(time)):
        ax.axvline(tc, color=ACCENT, lw=1.2, alpha=0.9)
        ax.axvspan(tc - duration / 2, tc + duration / 2,
                   color=ACCENT, alpha=0.12)

    ax.set_ylabel("Fluxo norm.")
    ax.set_title(f"Setor {s}", loc="left", fontweight="bold",
                 color=POINT, fontsize=12)
    ax.margins(x=0.005)

axes[-1].set_xlabel("Tempo [BTJD]")

fig.suptitle("TOI 3326.01  ·  TIC 301455423  ·  TESS SPOC",
             fontsize=15, fontweight="bold", color=POINT, y=0.995)

fig.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

# ==========================================================
# FIGURA 2 - PHASE FOLD (TODOS JUNTOS)
# ==========================================================

lc = LightCurveCollection(list(lc_dict.values())).stitch()

lc_fold = lc.fold(period=P, epoch_time=T0)
phase_h = lc_fold.phase.value * 24.0
flux_fold = lc_fold.flux.value

# binagem para a curva média
order = np.argsort(phase_h)
ph = phase_h[order]
fx = flux_fold[order]

nbins = 80
bins = np.linspace(-6, 6, nbins + 1)
idx = np.digitize(ph, bins)
bin_c, bin_f = [], []
for b in range(1, nbins + 1):
    m = idx == b
    if np.any(m):
        bin_c.append(0.5 * (bins[b - 1] + bins[b]))
        bin_f.append(np.median(fx[m]))

fig2, ax = plt.subplots(figsize=(11, 6))

ax.plot(phase_h, flux_fold, ".", color=POINT, ms=2, alpha=0.25,
        label="Pontos")
ax.plot(bin_c, bin_f, "-", color=ACCENT, lw=2.2,
        label="Mediana binada")

ax.axvline(0, color=ACCENT, lw=1, ls="--", alpha=0.6)
ax.axvspan(-duration_hours / 2, +duration_hours / 2,
           color=ACCENT, alpha=0.10)

ax.set_xlim(-6, 6)
ax.set_xlabel("Horas desde o centro do trânsito")
ax.set_ylabel("Fluxo normalizado")
ax.set_title(f"Phase Fold  ·  Setores {setores}",
             fontweight="bold", color=POINT, fontsize=13)
ax.legend(frameon=False)

fig2.tight_layout()
plt.show()

In [15]:
import warnings
warnings.simplefilter("ignore")

import numpy as np
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve, LightCurveCollection

%matplotlib qt

# ==========================================================
# ESTILO
# ==========================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.facecolor": "#fafafa",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.8,
    "axes.axisbelow": True,
    "figure.facecolor": "white",
})

ACCENT = "#e63946"
POINT  = "#1d3557"

# ==========================================================
# ALVO / EFEMÉRIDE
# ==========================================================

target = "TIC 301455423"

T0_BJD = 2459387.102253
P = 1.0664194                 # dias
duration_hours = 2.106        # horas

T0 = T0_BJD - 2457000
duration = duration_hours / 24.0

setores = [100, 101, 102, 103]

# ==========================================================
# BUSCA / DOWNLOAD
# ==========================================================

search = search_lightcurve(target, mission="TESS", author="SPOC")

lc_list = []

for s in setores:

    indice = None
    for i, row in enumerate(search.table):
        if (f"Sector {s}" in row["mission"]) and (row["exptime"] == 120):
            indice = i
            break

    if indice is None:
        print(f"Setor {s} não encontrado.")
        continue

    lc_s = search[indice].download()
    lc_s = lc_s.remove_nans()
    lc_s = lc_s.remove_outliers()
    lc_s = lc_s[lc_s.quality == 0]
    lc_s = lc_s.normalize().remove_nans()

    lc_list.append(lc_s)

if len(lc_list) == 0:
    raise Exception("Nenhuma curva encontrada.")

lc = LightCurveCollection(lc_list).stitch()

# ==========================================================
# PHASE FOLD - FASE NORMALIZADA (0 a 1), TRÂNSITO NO MEIO
# ==========================================================

lc_fold = lc.fold(period=P, epoch_time=T0, normalize_phase=True)

# desloca para o trânsito ficar em 0.5 (meio)
phase = (lc_fold.phase.value + 0.5) % 1.0
flux_fold = lc_fold.flux.value

# largura do trânsito em unidades de fase
dur_phase = duration / P

# ==========================================================
# BINAGEM (mediana)
# ==========================================================

order = np.argsort(phase)
ph = phase[order]
fx = flux_fold[order]

nbins = 120
bins = np.linspace(0, 1, nbins + 1)
idx = np.digitize(ph, bins)
bin_c, bin_f = [], []
for b in range(1, nbins + 1):
    m = idx == b
    if np.any(m):
        bin_c.append(0.5 * (bins[b - 1] + bins[b]))
        bin_f.append(np.median(fx[m]))

# ==========================================================
# GRÁFICO
# ==========================================================

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(phase, flux_fold, ".", color=POINT, ms=2, alpha=0.25, label="Pontos")
ax.plot(bin_c, bin_f, "-", color=ACCENT, lw=2.2, label="Mediana binada")

ax.axvline(0.5, color=ACCENT, lw=1, ls="--", alpha=0.6)
ax.axvspan(0.5 - dur_phase / 2, 0.5 + dur_phase / 2,
           color=ACCENT, alpha=0.10)

ax.set_xlim(0, 1)
ax.set_xlabel("Fase")
ax.set_ylabel("Fluxo normalizado")
ax.set_title(f"Phase Fold  ·  Setores {setores}",
             fontweight="bold", color=POINT, fontsize=13)
ax.legend(frameon=False)

fig.tight_layout()
plt.show()

In [16]:
import warnings
warnings.simplefilter("ignore")

import numpy as np
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve, LightCurveCollection

%matplotlib qt

# ==========================================================
# ESTILO
# ==========================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.facecolor": "#fafafa",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.8,
    "axes.axisbelow": True,
    "figure.facecolor": "white",
})

ACCENT = "#e63946"
POINT  = "#1d3557"

# ==========================================================
# ALVO / EFEMÉRIDE
# ==========================================================

target = "TIC 301455423"

T0_BJD = 2459387.102253
P = 1.0664194                 # dias
duration_hours = 2.106        # horas

T0 = T0_BJD - 2457000
duration = duration_hours / 24.0

setores = [100, 101, 102, 103]

# ==========================================================
# BUSCA / DOWNLOAD
# ==========================================================

search = search_lightcurve(target, mission="TESS", author="SPOC")

lc_list = []

for s in setores:

    indice = None
    for i, row in enumerate(search.table):
        if (f"Sector {s}" in row["mission"]) and (row["exptime"] == 120):
            indice = i
            break

    if indice is None:
        print(f"Setor {s} não encontrado.")
        continue

    lc_s = search[indice].download()
    lc_s = lc_s.remove_nans()
    lc_s = lc_s.remove_outliers()
    lc_s = lc_s[lc_s.quality == 0]
    lc_s = lc_s.normalize().remove_nans()

    lc_list.append(lc_s)

if len(lc_list) == 0:
    raise Exception("Nenhuma curva encontrada.")

lc = LightCurveCollection(lc_list).stitch()

# ==========================================================
# PHASE FOLD - TRÂNSITO NA FASE 0, CENTRADO NO MEIO
# ==========================================================

lc_fold = lc.fold(period=P, epoch_time=T0, normalize_phase=True)

phase = lc_fold.phase.value        # já vai de -0.5 a +0.5, trânsito em 0
flux_fold = lc_fold.flux.value

# largura do trânsito em unidades de fase
dur_phase = duration / P

# ==========================================================
# BINAGEM (mediana)
# ==========================================================

order = np.argsort(phase)
ph = phase[order]
fx = flux_fold[order]

nbins = 120
bins = np.linspace(-0.5, 0.5, nbins + 1)
idx = np.digitize(ph, bins)
bin_c, bin_f = [], []
for b in range(1, nbins + 1):
    m = idx == b
    if np.any(m):
        bin_c.append(0.5 * (bins[b - 1] + bins[b]))
        bin_f.append(np.median(fx[m]))

# ==========================================================
# GRÁFICO
# ==========================================================

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(phase, flux_fold, ".", color=POINT, ms=2, alpha=0.25, label="Pontos")
ax.plot(bin_c, bin_f, "-", color=ACCENT, lw=2.2, label="Mediana binada")

ax.axvline(0, color=ACCENT, lw=1, ls="--", alpha=0.6)
ax.axvspan(-dur_phase / 2, +dur_phase / 2,
           color=ACCENT, alpha=0.10)

ax.set_xlim(-0.5, 0.5)
ax.set_xlabel("Fase")
ax.set_ylabel("Fluxo normalizado")
ax.set_title(f"Phase Fold  ·  Setores {setores}",
             fontweight="bold", color=POINT, fontsize=13)
ax.legend(frameon=False)

fig.tight_layout()
plt.show()

In [17]:
import warnings
warnings.simplefilter("ignore")

import numpy as np
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve, LightCurveCollection

%matplotlib qt

# ==========================================================
# ESTILO
# ==========================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.facecolor": "#fafafa",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.8,
    "axes.axisbelow": True,
    "figure.facecolor": "white",
})

ACCENT = "#457b9d"      # azul p/ eclipse secundário
POINT  = "#1d3557"

# ==========================================================
# ALVO / EFEMÉRIDE
# ==========================================================

target = "TIC 301455423"

T0_BJD = 2459387.102253
P = 1.0664194                 # dias
duration_hours = 2.106        # horas

T0 = T0_BJD - 2457000
duration = duration_hours / 24.0

setores = [100, 101, 102, 103]

# ==========================================================
# BUSCA / DOWNLOAD
# ==========================================================

search = search_lightcurve(target, mission="TESS", author="SPOC")

lc_list = []

for s in setores:

    indice = None
    for i, row in enumerate(search.table):
        if (f"Sector {s}" in row["mission"]) and (row["exptime"] == 120):
            indice = i
            break

    if indice is None:
        print(f"Setor {s} não encontrado.")
        continue

    lc_s = search[indice].download()
    lc_s = lc_s.remove_nans()
    lc_s = lc_s.remove_outliers()
    lc_s = lc_s[lc_s.quality == 0]
    lc_s = lc_s.normalize().remove_nans()

    lc_list.append(lc_s)

if len(lc_list) == 0:
    raise Exception("Nenhuma curva encontrada.")

lc = LightCurveCollection(lc_list).stitch()

# ==========================================================
# PHASE FOLD - ECLIPSE SECUNDÁRIO NO MEIO (FASE 0.5 -> 0)
# ==========================================================

lc_fold = lc.fold(period=P, epoch_time=T0, normalize_phase=True)

# desloca 0.5 em fase: a ocultação (fase 0.5) vai para o centro (0)
phase = ((lc_fold.phase.value + 1.0) % 1.0) - 0.5
flux_fold = lc_fold.flux.value

# largura esperada do eclipse em unidades de fase
dur_phase = duration / P

# ==========================================================
# BINAGEM (mediana)
# ==========================================================

order = np.argsort(phase)
ph = phase[order]
fx = flux_fold[order]

nbins = 120
bins = np.linspace(-0.5, 0.5, nbins + 1)
idx = np.digitize(ph, bins)
bin_c, bin_f = [], []
for b in range(1, nbins + 1):
    m = idx == b
    if np.any(m):
        bin_c.append(0.5 * (bins[b - 1] + bins[b]))
        bin_f.append(np.median(fx[m]))

# ==========================================================
# GRÁFICO
# ==========================================================

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(phase, flux_fold, ".", color=POINT, ms=2, alpha=0.25, label="Pontos")
ax.plot(bin_c, bin_f, "-", color=ACCENT, lw=2.2, label="Mediana binada")

ax.axvline(0, color=ACCENT, lw=1, ls="--", alpha=0.6)
ax.axvspan(-dur_phase / 2, +dur_phase / 2,
           color=ACCENT, alpha=0.10)

ax.set_xlim(-0.5, 0.5)
ax.set_xlabel("Fase (0 = eclipse secundário)")
ax.set_ylabel("Fluxo normalizado")
ax.set_title(f"Eclipse secundário  ·  Setores {setores}",
             fontweight="bold", color=POINT, fontsize=13)
ax.legend(frameon=False)

fig.tight_layout()
plt.show()

In [23]:
import warnings
warnings.simplefilter("ignore")

import numpy as np
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve, LightCurveCollection

%matplotlib qt

# ==========================================================
# ESTILO
# ==========================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.facecolor": "#fafafa",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.8,
    "axes.axisbelow": True,
    "figure.facecolor": "white",
})

ACCENT = "#457b9d"      # azul p/ eclipse secundário
POINT  = "#1d3557"

# ==========================================================
# ALVO / EFEMÉRIDE
# ==========================================================

target = "TIC 301455423"

T0_BJD = 2459387.102253
P = 1.0664194                 # dias
duration_hours = 2.106        # horas

T0 = T0_BJD - 2457000
duration = duration_hours / 24.0

setores = [100, 101, 102, 103]

# ==========================================================
# BUSCA / DOWNLOAD
# ==========================================================

search = search_lightcurve(target, mission="TESS", author="SPOC")

lc_list = []

for s in setores:

    indice = None
    for i, row in enumerate(search.table):
        if (f"Sector {s}" in row["mission"]) and (row["exptime"] == 120):
            indice = i
            break

    if indice is None:
        print(f"Setor {s} não encontrado.")
        continue

    lc_s = search[indice].download()
    lc_s = lc_s.remove_nans()
    lc_s = lc_s.remove_outliers()
    lc_s = lc_s[lc_s.quality == 0]
    lc_s = lc_s.normalize().remove_nans()

    lc_list.append(lc_s)

if len(lc_list) == 0:
    raise Exception("Nenhuma curva encontrada.")

lc = LightCurveCollection(lc_list).stitch()

# ==========================================================
# PHASE FOLD - EIXO 0 A 1, SECUNDÁRIO NO MEIO (FASE 0.5)
# ==========================================================

lc_fold = lc.fold(period=P, epoch_time=T0, normalize_phase=True)

# fase de 0 a 1: primário nas bordas (0 e 1), secundário no meio (0.5)
phase = lc_fold.phase.value % 1.0
flux_fold = lc_fold.flux.value

# largura esperada do eclipse em unidades de fase
dur_phase = duration / P

# ==========================================================
# BINAGEM (mediana)
# ==========================================================

order = np.argsort(phase)
ph = phase[order]
fx = flux_fold[order]

nbins = 120
bins = np.linspace(0, 1, nbins + 1)
idx = np.digitize(ph, bins)
bin_c, bin_f = [], []
for b in range(1, nbins + 1):
    m = idx == b
    if np.any(m):
        bin_c.append(0.5 * (bins[b - 1] + bins[b]))
        bin_f.append(np.median(fx[m]))

# ==========================================================
# GRÁFICO
# ==========================================================

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(phase, flux_fold, ".", color=POINT, ms=2, alpha=0.25, label="Pontos")
ax.plot(bin_c, bin_f, "-", color="darkred", lw=2.2, label="Mediana binada")

# secundário no meio (0.5)
ax.axvline(0.5, color=ACCENT, lw=1, ls="--", alpha=0.6)
ax.axvspan(0.5 - dur_phase / 2, 0.5 + dur_phase / 2,
           color=ACCENT, alpha=0.10)

ax.set_xlim(0.25, 0.75)
ax.set_xlabel("Fase  (0.5 = eclipse secundário)")
ax.set_ylabel("Fluxo normalizado")
ax.set_title(f"Eclipse secundário  ·  Setores {setores}",
             fontweight="bold", color=POINT, fontsize=13)
ax.legend(frameon=False)

fig.tight_layout()
plt.show()

In [ ]:
import warnings
warnings.simplefilter("ignore")

import numpy as np
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve, LightCurveCollection

%matplotlib qt

# ==========================================================
# ESTILO
# ==========================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.facecolor": "#fafafa",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.8,
    "axes.axisbelow": True,
    "figure.facecolor": "white",
})

PRIMARY = "#e63946"     # vermelho p/ primário
ACCENT  = "#457b9d"     # azul p/ secundário
POINT   = "#1d3557"

# ==========================================================
# ALVO / EFEMÉRIDE
# ==========================================================

target = "TIC 301455423"

T0_BJD = 2459387.102253
P = 1.0664194                 # dias
duration_hours = 2.106        # horas

T0 = T0_BJD - 2457000
duration = duration_hours / 24.0

setores = [100, 101, 102, 103]

# posições de fase dos eclipses
fase_primario   = 1.0
fase_secundario = 0.25

# ==========================================================
# BUSCA / DOWNLOAD
# ==========================================================

search = search_lightcurve(target, mission="TESS", author="SPOC")

lc_list = []

for s in setores:

    indice = None
    for i, row in enumerate(search.table):
        if (f"Sector {s}" in row["mission"]) and (row["exptime"] == 120):
            indice = i
            break

    if indice is None:
        print(f"Setor {s} não encontrado.")
        continue

    lc_s = search[indice].download()
    lc_s = lc_s.remove_nans()
    lc_s = lc_s.remove_outliers()
    lc_s = lc_s[lc_s.quality == 0]
    lc_s = lc_s.normalize().remove_nans()

    lc_list.append(lc_s)

if len(lc_list) == 0:
    raise Exception("Nenhuma curva encontrada.")

lc = LightCurveCollection(lc_list).stitch()

# ==========================================================
# PHASE FOLD - EIXO 0 A 1, PRIMÁRIO NAS BORDAS (0 e 1)
# ==========================================================

lc_fold = lc.fold(period=P, epoch_time=T0, normalize_phase=True)

phase = lc_fold.phase.value % 1.0     # 0 a 1, primário em 0 e 1
flux_fold = lc_fold.flux.value

dur_phase = duration / P

# ==========================================================
# BINAGEM (mediana)
# ==========================================================

order = np.argsort(phase)
ph = phase[order]
fx = flux_fold[order]

nbins = 120
bins = np.linspace(0, 1, nbins + 1)
idx = np.digitize(ph, bins)
bin_c, bin_f = [], []
for b in range(1, nbins + 1):
    m = idx == b
    if np.any(m):
        bin_c.append(0.5 * (bins[b - 1] + bins[b]))
        bin_f.append(np.median(fx[m]))

# ==========================================================
# GRÁFICO
# ==========================================================

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(phase, flux_fold, ".", color=POINT, ms=2, alpha=0.25, label="Pontos")
ax.plot(bin_c, bin_f, "-", color=ACCENT, lw=2.2, label="Mediana binada")

# primário nas bordas (0 e 1)
for xp in (0.0, fase_primario):
    ax.axvline(xp, color=PRIMARY, lw=1, ls="--", alpha=0.7)
    ax.axvspan(xp - dur_phase / 2, xp + dur_phase / 2,
               color=PRIMARY, alpha=0.10)

# secundário em 0.25
ax.axvline(fase_secundario, color=ACCENT, lw=1, ls="--", alpha=0.7)
ax.axvspan(fase_secundario - dur_phase / 2, fase_secundario + dur_phase / 2,
           color=ACCENT, alpha=0.12)

ax.set_xlim(0, 1)
ax.set_xlabel("Fase")
ax.set_ylabel("Fluxo normalizado")
ax.set_title(f"Primário (fase 0/1)  ·  Secundário (fase {fase_secundario})  ·  Setores {setores}",
             fontweight="bold", color=POINT, fontsize=12)
ax.legend(frameon=False)

fig.tight_layout()
plt.show()

: 